In [4]:
# Cellule 1 — Connexion
from azure.ai.ml import MLClient, Input
from azure.ai.ml.constants import AssetTypes
from azure.identity import DefaultAzureCredential
import sys, os, datetime, json
 
ml_client = MLClient.from_config(DefaultAzureCredential())
print(f'✓ Workspace : {ml_client.workspace_name}')

Found the config file in: ./config.json


✓ Workspace : ws-pauline


In [3]:
from azure.identity import InteractiveBrowserCredential

try:
    credential = DefaultAzureCredential()
    # Check if given credential can get token successfully.
    credential.get_token("https://management.azure.com/.default")
except :
    # Fall back to InteractiveBrowserCredential in case DefaultAzureCredential not work
    # This will open a browser page for
    credential = InteractiveBrowserCredential()

DefaultAzureCredential failed to retrieve a token from the included credentials.
Attempted credentials:
	EnvironmentCredential: EnvironmentCredential authentication unavailable. Environment variables are not fully configured.
Visit https://aka.ms/azsdk/python/identity/environmentcredential/troubleshoot to troubleshoot this issue.
	ManagedIdentityCredential: Unexpected response "{'error': 'SSO failure, to mitigated it please try to click Jupyter/JupyterLab.'}"
Content: {"error":"SSO failure, to mitigated it please try to click Jupyter/JupyterLab."}
To mitigate this issue, please refer to the troubleshooting guidelines here at https://aka.ms/azsdk/python/identity/defaultazurecredential/troubleshoot.


In [ ]:
# Cellule 2 — Créer ou mettre à jour l'Environment (si conda.yml a changé)
# ⚠️  Incrémenter ENV_VERSION à chaque modification du conda.yml
from azure.ai.ml.entities import Environment

PROJECT_ROOT = os.path.expanduser('~/cloudfiles/code/diabetes-pipeline')
CONDA_PATH = os.path.join(PROJECT_ROOT, 'conda.yml')
 
ENV_VERSION = '1'  # → passer à '2', '3'... si conda.yml change
 
env = Environment(
    name='diabetes-ml-env',
    version=ENV_VERSION,
    conda_file=CONDA_PATH,
    image='mcr.microsoft.com/azureml/openmpi4.1.0-ubuntu20.04:latest',
)
env = ml_client.environments.create_or_update(env)
print(f'Environment : {env.name} v{env.version}')

Environment : diabetes-ml-env v2


In [7]:
# Cellule 3 — Importer les composants

os.chdir(os.path.expanduser('~/cloudfiles/code/diabetes-pipeline'))
project_root = os.path.abspath('')
if project_root not in sys.path:
    sys.path.insert(0, project_root)
print(f'Dossier projet : {project_root}')
 
from src.preprocess          import preprocess_diabetes
from src.feature_engineering import feature_engineering_diabetes
from src.train               import train_diabetes
from src.evaluate            import evaluate_diabetes
from pipeline import diabetes_pipeline
print('✓ Tous les composants importés')

Dossier projet : /mnt/batch/tasks/shared/LS_root/mounts/clusters/instance-pauline/code/diabetes-pipeline
✓ Tous les composants importés


In [8]:
# Cellule 4 — Soumettre le job
asset = ml_client.data.get('diabetes-dataset', version='1')
 
ALPHA = 1.0   # essayer aussi 0.1, 10.0, 100.0
 
pipeline_job = diabetes_pipeline(
    raw_data=Input(type=AssetTypes.URI_FILE, path=asset.path),
    alpha=ALPHA,
)
ts = datetime.datetime.now().strftime('%m%d-%H%M')
pipeline_job.experiment_name = 'diabetes-progression'
pipeline_job.display_name    = f'ridge-alpha{ALPHA}-{ts}'
 
submitted = ml_client.jobs.create_or_update(pipeline_job)
print(f'Job soumis : {submitted.name}')
print(f'URL Studio : {submitted.studio_url}')

Class AutoDeleteSettingSchema: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.
Class AutoDeleteConditionSchema: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.
Class BaseAutoDeleteSettingSchema: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.
Class IntellectualPropertySchema: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.
Class ProtectionLevelSchema: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.
Class BaseIntellectualPropertySchema: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.
Uploading diabetes-pipeline (0.05 MBs)

Job soumis : frosty_fly_j383kvfps7
URL Studio : https://ml.azure.com/runs/frosty_fly_j383kvfps7?wsid=/subscriptions/b1195388-942c-4df6-8b0e-4b4daee48d5e/resourcegroups/rg-mlops-pauline/workspaces/ws-pauline&tid=f017b958-d19a-4cb1-b4e7-03a430980b51


In [9]:
# Cellule 5 — Lire les métriques (après Completed)
job = ml_client.jobs.get(submitted.name)
print(f'Statut : {job.status}')
 
if job.status == 'Completed':
    out_dir = os.path.expanduser(
        f'~/cloudfiles/code/diabetes-pipeline/outputs/{submitted.name}')
    os.makedirs(out_dir, exist_ok=True)
    ml_client.jobs.download(
        name=submitted.name, download_path=out_dir, output_name='metrics')
    with open(f'{out_dir}/named-outputs/metrics/metrics.json') as f:
        m = json.load(f)
    print(f'MAE  : {m["mae"]:.2f}')
    print(f'RMSE : {m["rmse"]:.2f}')
    print(f'R²   : {m["r2"]:.4f}')

Statut : Completed


MAE  : 42.81
RMSE : 53.78
R²   : 0.4541
